# Exploring Pooled Testing Decision Trees

This notebook provides a visual, interactive exploration of decision trees for
**Dynamic Augmented Pooled Testing Strategies (DAPTS)**.

We start with tiny pedagogical examples (n=2, 3, 4) to build intuition about
how prevalence, value, budget, and pool size shape the optimal strategy, then
scale up to richer scenarios involving heterogeneous populations, greedy vs
optimal comparisons, hybrid solvers, and infection-aware scoring.

**Blocks:**
- **Block 0** -- Building intuition with simple trees
- **Block 1** -- Two worlds: VIP vs common populations
- **Block 2** -- Binary search strategies for 8 valuable people
- **Block 3** -- Greedy vs optimal grid comparison
- **Block 4** -- Hybrid greedy-to-brute-force solver
- **Block 5** -- Infection-aware greedy scoring

In [ ]:
import sys, os, time
# Add repo root to path (two levels up from notebooks/)
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(''))))

%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from IPython.display import display, HTML

from augmented.core import (
    mask_from_indices, indices_from_mask, mask_str,
    all_pools, all_pools_from_mask, compute_active_mask, popcount,
)
from augmented.bayesian import (
    bayesian_update_single_test, _poisson_binomial_pmf,
)
from augmented.solver import solve_optimal_dapts
from augmented.classical_solver import solve_classical_dynamic
from augmented.greedy import (
    greedy_myopic_expected_utility,
    greedy_myopic_counting_expected_utility,
    _myopic_best_pool,
)
from augmented.baselines import u_max, u_single
from augmented.tree_extractor import (
    extract_tree, print_tree, summarize_tree, prune_tree,
)
from augmented.tree_visualizer import (
    render_tree, render_side_by_side, render_tree_series,
)
from augmented.hybrid_solver import (
    hybrid_greedy_bruteforce,
    estimate_branch_value,
    expected_info_gain,
    infection_aware_score,
    _infection_aware_best_pool,
)

matplotlib.rcParams['figure.figsize'] = (10, 6)
matplotlib.rcParams['font.size'] = 12
print('All imports OK')

---
## Block 0: Building Intuition -- Simple Pedagogical Trees

We start with the smallest possible instances so the full decision tree fits
on screen and every branch can be understood at a glance.

### 0.1 The Simplest Case -- n=2, B=1, G=2

Two people, one test, equal infection probability p=0.2, unit utility.

We compare two strategies:
- **Strategy A**: test person 0 alone (pool size 1)
- **Strategy B**: pool both people together (pool size 2)

In [ ]:
# --- 0.1 Simplest Case: n=2, B=1, G=2 ---
n, B, G = 2, 1, 2
p = [0.2, 0.2]
u = [1.0, 1.0]

# Strategy A: test person 0 alone
# If r=0 (prob 0.8): clear person 0, utility = 1
# If r=1 (prob 0.2): clear nobody, utility = 0
eu_A = 0.8 * 1.0 + 0.2 * 0.0

# Strategy B: pool {0,1}
# If r=0 (prob 0.8*0.8=0.64): clear both, utility = 2
# If r=1 or r=2: clear nobody, utility = 0
eu_B = 0.64 * 2.0

print(f'Strategy A (test {{0}} alone):  EU = {eu_A:.2f}')
print(f'Strategy B (pool {{0,1}}):      EU = {eu_B:.2f}')
print(f'Pooling wins by {eu_B - eu_A:.2f} expected utility.')
print()

# Solve optimally to confirm
val_opt, policy_opt = solve_optimal_dapts(p, u, B, G)
print(f'Optimal DP value: {val_opt:.4f}')

# Build the two trees manually for visualization
# Tree A: test {0}
pool_A = mask_from_indices([0])
tree_A = {
    'step': 1, 'terminal': False,
    'pool': pool_A, 'pool_str': mask_str(pool_A, n),
    'history': (), 'cleared': 0, 'cleared_str': mask_str(0, n),
    'posteriors': list(p),
    'children': {
        0: {'step': 2, 'terminal': True, 'history': ((pool_A, 0),),
            'cleared': pool_A, 'cleared_str': mask_str(pool_A, n),
            'posteriors': [0.0, 0.2], 'utility': 1.0},
        1: {'step': 2, 'terminal': True, 'history': ((pool_A, 1),),
            'cleared': 0, 'cleared_str': mask_str(0, n),
            'posteriors': [1.0, 0.2], 'utility': 0.0},
    }
}

# Tree B: test {0,1}
pool_B = mask_from_indices([0, 1])
tree_B = {
    'step': 1, 'terminal': False,
    'pool': pool_B, 'pool_str': mask_str(pool_B, n),
    'history': (), 'cleared': 0, 'cleared_str': mask_str(0, n),
    'posteriors': list(p),
    'children': {
        0: {'step': 2, 'terminal': True, 'history': ((pool_B, 0),),
            'cleared': pool_B, 'cleared_str': mask_str(pool_B, n),
            'posteriors': [0.0, 0.0], 'utility': 2.0},
        1: {'step': 2, 'terminal': True, 'history': ((pool_B, 1),),
            'cleared': 0, 'cleared_str': mask_str(0, n),
            'posteriors': [0.5, 0.5], 'utility': 0.0},
        2: {'step': 2, 'terminal': True, 'history': ((pool_B, 2),),
            'cleared': 0, 'cleared_str': mask_str(0, n),
            'posteriors': [1.0, 1.0], 'utility': 0.0},
    }
}

display(render_side_by_side(
    tree_A, tree_B, n,
    title_a=f'A: test {{0}} alone (EU={eu_A:.2f})',
    title_b=f'B: pool {{0,1}} (EU={eu_B:.2f})',
    show_posteriors=True,
))

### 0.2 The Value Effect -- n=2, B=1, G=2, Asymmetric Utilities

Person A has utility u=10 but p=0.1.  Person B has u=1, p=0.1.
When one person is far more valuable, the optimizer may prefer to test
that person alone (guaranteed information about the high-value target)
rather than pooling.

We sweep u_A from 1 to 20 to see where the transition happens.

In [ ]:
# --- 0.2 Value Effect: n=2, B=1, G=2, asymmetric ---
n, B, G = 2, 1, 2
p = [0.1, 0.1]
u_base = [10.0, 1.0]

val_opt, policy_opt = solve_optimal_dapts(p, u_base, B, G)
tree_opt = extract_tree(policy_opt, p, u_base, n)
print(f'Optimal tree for u_A=10, u_B=1 (EU = {val_opt:.4f}):')

group_colors = {0: 'red', 1: 'gray'}
display(render_tree(tree_opt, n, group_colors=group_colors,
                    title=f'Optimal: u_A=10, u_B=1 (EU={val_opt:.2f})'))
print()

In [ ]:
# Sweep u_A from 1 to 20
n, B, G = 2, 1, 2
p = [0.1, 0.1]

u_A_values = np.arange(1, 21, 1)
eu_values = []
first_pools = []  # which pool is chosen at step 1

for u_a in u_A_values:
    u_sweep = [float(u_a), 1.0]
    val, pol = solve_optimal_dapts(p, u_sweep, B, G)
    tree = extract_tree(pol, p, u_sweep, n)
    eu_values.append(val)
    first_pools.append(tree['pool_str'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(u_A_values, eu_values, 'o-', color='#2980b9', linewidth=2)
ax1.set_xlabel('u_A (utility of person A)')
ax1.set_ylabel('Expected Utility (optimal)')
ax1.set_title('EU vs u_A (p=[0.1, 0.1], B=1, G=2)')
ax1.grid(True, alpha=0.3)

# Color-code by action chosen
unique_pools = sorted(set(first_pools))
pool_to_color = {}
cmap = plt.cm.Set1
for idx, pool_str in enumerate(unique_pools):
    pool_to_color[pool_str] = cmap(idx / max(len(unique_pools) - 1, 1))

colors_bar = [pool_to_color[fp] for fp in first_pools]
ax2.bar(u_A_values, eu_values, color=colors_bar, edgecolor='black', alpha=0.8)
ax2.set_xlabel('u_A (utility of person A)')
ax2.set_ylabel('Expected Utility')
ax2.set_title('Optimal First Action vs u_A')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=pool_to_color[ps], edgecolor='black',
                         label=f'Test {ps}')
                   for ps in unique_pools]
ax2.legend(handles=legend_elements, title='First Pool')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Report transitions
for i in range(1, len(first_pools)):
    if first_pools[i] != first_pools[i-1]:
        print(f'Transition at u_A={u_A_values[i]:.0f}: '
              f'{first_pools[i-1]} -> {first_pools[i]}')

### 0.3 The Prevalence Effect -- n=3, B=2, G=3

Three equal people (u=1), sweeping infection probability
p in {0.05, 0.2, 0.5}.  Low prevalence favors large pools (likely all
clear in one test), high prevalence favors smaller pools (large pools
almost certainly fail).

In [ ]:
# --- 0.3 Prevalence Effect: n=3, B=2, G=3 ---
n, B, G = 3, 2, 3
u = [1.0, 1.0, 1.0]

p_values = [0.05, 0.2, 0.5]
trees = []
titles = []

for pv in p_values:
    p_inst = [pv] * n
    val, pol = solve_optimal_dapts(p_inst, u, B, G)
    tree = extract_tree(pol, p_inst, u, n)
    trees.append(tree)
    titles.append(f'p={pv} (EU={val:.3f})')
    print(f'p={pv}: EU={val:.4f}, first pool = {tree["pool_str"]}')

display(render_tree_series(trees, n, titles, max_per_row=3,
                           show_posteriors=False))

### 0.4 Augmented vs Classical -- n=3, B=2, G=3

The augmented test returns the exact count r = |pool intersect Z|, while the
classical test returns only positive/negative.  More information should
translate into higher expected utility.

Note: `solve_classical_dynamic` returns `(value, None)` -- no policy, so we
cannot extract a tree for the classical strategy.

In [ ]:
# --- 0.4 Augmented vs Classical ---
n, B, G = 3, 2, 3
p = [0.2, 0.25, 0.15]
u = [3.0, 2.0, 4.0]

val_aug, pol_aug = solve_optimal_dapts(p, u, B, G)
val_cls, _ = solve_classical_dynamic(p, u, B, G)

tree_aug = extract_tree(pol_aug, p, u, n)

print(f'Augmented optimal EU: {val_aug:.4f}')
print(f'Classical  optimal EU: {val_cls:.4f}')
print(f'Augmented advantage:  +{val_aug - val_cls:.4f} '
      f'(+{(val_aug - val_cls) / val_cls * 100:.2f}%)')
print()
print('Augmented decision tree:')

display(render_tree(tree_aug, n, title=f'Augmented Optimal (EU={val_aug:.3f})'))
print()
print('Note: classical tree cannot be shown because solve_classical_dynamic '
      'returns (value, None) -- no policy object.')

### 0.5 The Power of Budget -- n=4, B in {1,2,3,4}, G=4

Four equal people (p=0.15, u=1).  More tests = more opportunities to
clear people.  The expected utility approaches U_max as B grows.

In [ ]:
# --- 0.5 Power of Budget ---
n, G = 4, 4
p = [0.15] * n
u = [1.0] * n

B_values = [1, 2, 3, 4]
trees_b = []
titles_b = []
eus_b = []

for B in B_values:
    val, pol = solve_optimal_dapts(p, u, B, G)
    tree = extract_tree(pol, p, u, n)
    trees_b.append(tree)
    titles_b.append(f'B={B} (EU={val:.3f})')
    eus_b.append(val)
    print(f'B={B}: EU = {val:.4f}')

u_max_val = u_max(p, u)
print(f'U_max (infinite budget) = {u_max_val:.4f}')

display(render_tree_series(trees_b, n, titles_b, max_per_row=2,
                           show_posteriors=False))

In [ ]:
# Utility vs Budget curve
n, G = 4, 4
p = [0.15] * n
u = [1.0] * n

B_range = range(1, 7)
eus_curve = []
for B in B_range:
    val, _ = solve_optimal_dapts(p, u, B, G)
    eus_curve.append(val)

u_max_val = u_max(p, u)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(B_range), eus_curve, 'o-', color='#27ae60', linewidth=2,
        markersize=8, label='Optimal EU')
ax.axhline(y=u_max_val, color='#e74c3c', linestyle='--', linewidth=2,
           label=f'U_max = {u_max_val:.3f}')
ax.set_xlabel('Budget B (number of tests)')
ax.set_ylabel('Expected Utility')
ax.set_title(f'Utility vs Budget -- n={n}, p=0.15, u=1, G={G}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 0.6 "Why Not Test One by One?" -- n=4, B=2, G in {1,2,3,4}

Fixed budget B=2, varying maximum pool size G.  G=1 forces individual
testing; larger G unlocks pooling.  How much does pooling help?

In [ ]:
# --- 0.6 Why Not Test One by One? ---
n, B = 4, 2
p = [0.15] * n
u = [1.0] * n

G_values = [1, 2, 3, 4]
trees_g = []
titles_g = []

for G in G_values:
    val, pol = solve_optimal_dapts(p, u, B, G)
    tree = extract_tree(pol, p, u, n)
    trees_g.append(tree)
    titles_g.append(f'G={G} (EU={val:.3f})')
    print(f'G={G}: EU = {val:.4f}, first pool = {tree["pool_str"]}')

display(render_tree_series(trees_g, n, titles_g, max_per_row=2,
                           show_posteriors=False))

---
## Block 1: "Two Worlds" -- VIP vs Common

Population of n=10 split into two groups:
- **4 VIPs** (indices 0-3): high utility u=10, high prevalence p=0.3
- **6 common** (indices 4-9): low utility u=1, low prevalence p=0.1

Budget B=5, max pool G=4.  How does the optimizer allocate tests
across the two groups?

In [ ]:
# --- Block 1: Two Worlds ---
n = 10
B, G = 5, 4

p = [0.3]*4 + [0.1]*6
u = [10.0]*4 + [1.0]*6

group_colors = {i: 'red' for i in range(4)}
group_colors.update({i: 'gray' for i in range(4, 10)})

# Optimal (DP)
t0 = time.time()
val_opt, pol_opt = solve_optimal_dapts(p, u, B, G)
time_opt = time.time() - t0
tree_opt = extract_tree(pol_opt, p, u, n)

# Greedy
t0 = time.time()
tree_greedy, eu_greedy = hybrid_greedy_bruteforce(p, u, B, G,
                                                   greedy_steps=B)
time_greedy = time.time() - t0

print(f'Optimal DP:  EU = {val_opt:.4f}  (time: {time_opt:.3f}s)')
print(f'Greedy:      EU = {eu_greedy:.4f}  (time: {time_greedy:.3f}s)')
print(f'Gap: {val_opt - eu_greedy:.4f} ({(val_opt - eu_greedy)/val_opt*100:.2f}%)')
print()

# Utility breakdown by group
# For the optimal tree: sum utilities of VIPs and common at terminals
print('First pool (optimal):', tree_opt.get('pool_str', 'N/A'))
print('First pool (greedy): ', tree_greedy.get('pool_str', 'N/A'))

In [ ]:
# Side-by-side visualization (pruned for readability)
n = 10
group_colors = {i: 'red' for i in range(4)}
group_colors.update({i: 'gray' for i in range(4, 10)})

display(render_side_by_side(
    tree_opt, tree_greedy, n,
    title_a=f'Optimal (EU={val_opt:.2f})',
    title_b=f'Greedy (EU={eu_greedy:.2f})',
    group_colors=group_colors,
    show_posteriors=False,
    max_depth=3,
))

In [ ]:
# Sensitivity: sweep p_VIP
n = 10
B, G = 5, 4

p_vip_values = [0.1, 0.2, 0.3, 0.5]
eu_opt_list = []
eu_greedy_list = []

for p_vip in p_vip_values:
    p_inst = [p_vip]*4 + [0.1]*6
    u_inst = [10.0]*4 + [1.0]*6
    val, _ = solve_optimal_dapts(p_inst, u_inst, B, G)
    _, eu_gr = hybrid_greedy_bruteforce(p_inst, u_inst, B, G, greedy_steps=B)
    eu_opt_list.append(val)
    eu_greedy_list.append(eu_gr)
    print(f'p_VIP={p_vip:.1f}: optimal={val:.4f}, greedy={eu_gr:.4f}')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(p_vip_values, eu_opt_list, 'o-', color='#27ae60', linewidth=2,
        label='Optimal (DP)', markersize=8)
ax.plot(p_vip_values, eu_greedy_list, 's--', color='#e74c3c', linewidth=2,
        label='Greedy', markersize=8)
ax.set_xlabel('p_VIP (infection probability of VIPs)')
ax.set_ylabel('Expected Utility')
ax.set_title(f'Two Worlds Sensitivity: EU vs p_VIP (n=10, B={B}, G={G})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Block 2: "Binary Search" -- 8 Valuable People

n=8 agents, all with u=10 and p=0.15.  Budget B=10, pool size G=8.

A natural strategy is **binary search**: test the whole group, and if
infected, split in half and recurse.  We compare this hand-crafted
strategy against the optimal DP and greedy.

In [ ]:
# --- Block 2: Binary Search Strategy ---

def build_binary_search_tree(agents, p, u, n, B, step, cleared_mask, history):
    """Recursively build a binary-search-style decision tree.
    
    Tests all agents in the group. If r=0, clear all.
    If r>0, split in half and recurse on the left half first.
    """
    if step > B or not agents:
        utility = sum(u[i] for i in indices_from_mask(cleared_mask, n))
        return {
            'step': step, 'terminal': True,
            'history': history, 'cleared': cleared_mask,
            'cleared_str': mask_str(cleared_mask, n),
            'posteriors': list(p), 'utility': utility,
        }
    
    pool = mask_from_indices(agents)
    pool_probs = [p[i] for i in agents]
    pmf = _poisson_binomial_pmf(pool_probs)
    
    children = {}
    
    # r=0: all clear
    if pmf[0] > 1e-15:
        new_cleared = cleared_mask | pool
        new_p = bayesian_update_single_test(list(p), pool, 0, n)
        new_hist = history + ((pool, 0),)
        utility = sum(u[i] for i in indices_from_mask(new_cleared, n))
        children[0] = {
            'step': step + 1, 'terminal': True,
            'history': new_hist, 'cleared': new_cleared,
            'cleared_str': mask_str(new_cleared, n),
            'posteriors': new_p, 'utility': utility,
        }
    
    # r>0: split and recurse on left half
    for r in range(1, len(agents) + 1):
        if pmf[r] < 1e-15:
            continue
        new_p = bayesian_update_single_test(list(p), pool, r, n)
        new_hist = history + ((pool, r),)
        
        if len(agents) <= 1:
            # Single agent, r>0 means infected
            utility = sum(u[i] for i in indices_from_mask(cleared_mask, n))
            children[r] = {
                'step': step + 1, 'terminal': True,
                'history': new_hist, 'cleared': cleared_mask,
                'cleared_str': mask_str(cleared_mask, n),
                'posteriors': new_p, 'utility': utility,
            }
        else:
            # Split in half, recurse on left half
            mid = len(agents) // 2
            left_half = agents[:mid]
            children[r] = build_binary_search_tree(
                left_half, new_p, u, n, B,
                step + 1, cleared_mask, new_hist
            )
    
    return {
        'step': step, 'terminal': False,
        'pool': pool, 'pool_str': mask_str(pool, n),
        'history': history, 'cleared': cleared_mask,
        'cleared_str': mask_str(cleared_mask, n),
        'posteriors': list(p), 'children': children,
    }


n = 8
B, G = 10, 8
p = [0.15] * n
u = [10.0] * n

# Binary search tree
agents_all = list(range(n))
tree_binsearch = build_binary_search_tree(
    agents_all, p, u, n, B, step=1, cleared_mask=0, history=()
)

# P(all clear in 1 test)
p_all_clear = 1.0
for pi in p:
    p_all_clear *= (1.0 - pi)
print(f'P(all 8 clear in 1 test) = {p_all_clear:.4f}')

# Optimal (DP)
t0 = time.time()
val_opt, pol_opt = solve_optimal_dapts(p, u, B, G)
time_opt = time.time() - t0
tree_opt = extract_tree(pol_opt, p, u, n)

# Greedy
t0 = time.time()
tree_greedy, eu_greedy = hybrid_greedy_bruteforce(p, u, B, G, greedy_steps=B)
time_greedy = time.time() - t0

print(f'\nOptimal DP:     EU = {val_opt:.4f}  ({time_opt:.2f}s)')
print(f'Greedy:         EU = {eu_greedy:.4f}  ({time_greedy:.2f}s)')

# Summarize the binary search tree
stats_bs = summarize_tree(tree_binsearch, n)
print(f'Binary search:  {stats_bs["total_nodes"]} nodes, '
      f'max depth {stats_bs["max_depth"]}')

In [ ]:
# Show all 3 trees (pruned for readability)
n = 8

display(render_tree_series(
    [tree_binsearch, tree_opt, tree_greedy], n,
    titles=['Binary Search', f'Optimal (EU={val_opt:.2f})',
            f'Greedy (EU={eu_greedy:.2f})'],
    max_per_row=3,
    show_posteriors=False,
    max_depth=3,
))

---
## Block 3: Greedy vs Optimal Grid

We generate a grid of problem instances varying n, B, and G, then compare
optimal (DP) vs greedy expected utility.  The gap tells us when exact
optimization matters vs when greedy is "good enough."

We also compare two greedy variants: sequential (standard Bayesian updates)
and counting (full-history updates).

In [ ]:
# --- Block 3: Greedy vs Optimal Grid ---
np.random.seed(42)

n_values = [5, 6, 7, 8]
B_values = [3, 4, 5]
G_values = [3, 4]

results_grid = []  # list of dicts

for G in G_values:
    for n in n_values:
        for B in B_values:
            p_inst = np.random.uniform(0.05, 0.4, size=n).tolist()
            u_inst = np.random.uniform(1, 10, size=n).tolist()
            
            t0 = time.time()
            val_opt, pol_opt = solve_optimal_dapts(p_inst, u_inst, B, G)
            time_opt = time.time() - t0
            
            eu_seq = greedy_myopic_expected_utility(p_inst, u_inst, B, G)
            eu_cnt = greedy_myopic_counting_expected_utility(
                p_inst, u_inst, B, G)
            
            gap_seq = val_opt - eu_seq
            gap_cnt = val_opt - eu_cnt
            gap_pct = gap_seq / val_opt * 100 if val_opt > 0 else 0
            
            results_grid.append({
                'n': n, 'B': B, 'G': G,
                'opt': val_opt, 'greedy_seq': eu_seq,
                'greedy_cnt': eu_cnt,
                'gap_seq': gap_seq, 'gap_cnt': gap_cnt,
                'gap_pct': gap_pct,
                'time_opt': time_opt,
                'p': p_inst, 'u': u_inst,
            })
            print(f'n={n}, B={B}, G={G}: opt={val_opt:.4f}, '
                  f'greedy_seq={eu_seq:.4f}, greedy_cnt={eu_cnt:.4f}, '
                  f'gap={gap_pct:.2f}%, time={time_opt:.2f}s')

In [ ]:
# Heatmap of optimality gaps
fig, axes = plt.subplots(1, len(G_values), figsize=(7*len(G_values), 5))
if len(G_values) == 1:
    axes = [axes]

for g_idx, G in enumerate(G_values):
    ax = axes[g_idx]
    subset = [r for r in results_grid if r['G'] == G]
    
    # Build matrix: rows = n, cols = B
    gap_matrix = np.zeros((len(n_values), len(B_values)))
    for r in subset:
        i = n_values.index(r['n'])
        j = B_values.index(r['B'])
        gap_matrix[i, j] = r['gap_pct']
    
    im = ax.imshow(gap_matrix, cmap='YlOrRd', aspect='auto',
                   vmin=0, vmax=max(r['gap_pct'] for r in results_grid))
    ax.set_xticks(range(len(B_values)))
    ax.set_xticklabels([f'B={b}' for b in B_values])
    ax.set_yticks(range(len(n_values)))
    ax.set_yticklabels([f'n={nv}' for nv in n_values])
    ax.set_title(f'Greedy Optimality Gap (%) -- G={G}')
    
    for i in range(len(n_values)):
        for j in range(len(B_values)):
            ax.text(j, i, f'{gap_matrix[i,j]:.1f}%',
                    ha='center', va='center', fontsize=10,
                    color='white' if gap_matrix[i,j] > 3 else 'black')
    
    plt.colorbar(im, ax=ax, label='Gap (%)')

plt.tight_layout()
plt.show()

In [ ]:
# Worst-gap case: show trees side-by-side
worst = max(results_grid, key=lambda r: r['gap_pct'])
print(f'Worst gap: n={worst["n"]}, B={worst["B"]}, G={worst["G"]}, '
      f'gap={worst["gap_pct"]:.2f}%')
print(f'  p = {["{:.3f}".format(pi) for pi in worst["p"]]}')
print(f'  u = {["{:.1f}".format(ui) for ui in worst["u"]]}')

p_w, u_w = worst['p'], worst['u']
n_w, B_w, G_w = worst['n'], worst['B'], worst['G']

_, pol_w = solve_optimal_dapts(p_w, u_w, B_w, G_w)
tree_opt_w = extract_tree(pol_w, p_w, u_w, n_w)
tree_greedy_w, eu_greedy_w = hybrid_greedy_bruteforce(
    p_w, u_w, B_w, G_w, greedy_steps=B_w)

display(render_side_by_side(
    tree_opt_w, tree_greedy_w, n_w,
    title_a=f'Optimal (EU={worst["opt"]:.3f})',
    title_b=f'Greedy (EU={eu_greedy_w:.3f})',
    show_posteriors=False,
    max_depth=3,
))

In [ ]:
# Compare greedy variants: sequential vs counting
fig, ax = plt.subplots(figsize=(8, 5))

seq_vals = [r['greedy_seq'] for r in results_grid]
cnt_vals = [r['greedy_cnt'] for r in results_grid]
opt_vals = [r['opt'] for r in results_grid]

labels_grid = [f'n={r["n"]},B={r["B"]},G={r["G"]}' for r in results_grid]

x = np.arange(len(results_grid))
width = 0.3

ax.bar(x - width, seq_vals, width, label='Greedy Sequential', color='#e74c3c',
       alpha=0.7, edgecolor='black')
ax.bar(x, cnt_vals, width, label='Greedy Counting', color='#8e44ad',
       alpha=0.7, edgecolor='black')
ax.bar(x + width, opt_vals, width, label='Optimal', color='#27ae60',
       alpha=0.7, edgecolor='black')

ax.set_xlabel('Instance')
ax.set_ylabel('Expected Utility')
ax.set_title('Greedy Sequential vs Counting vs Optimal')
ax.set_xticks(x)
ax.set_xticklabels(labels_grid, rotation=90, fontsize=7)
ax.legend()

plt.tight_layout()
plt.show()

---
## Block 4: Hybrid Greedy -> Brute Force

The hybrid solver uses greedy selection for the first K steps, then
switches to exact DP for the remaining B-K steps.  This lets us
trade off computation time vs solution quality.

We sweep K from 0 (full DP) to B (full greedy) and track both EU and
runtime.

In [ ]:
# --- Block 4: Hybrid Solver ---
np.random.seed(123)
n, B, G = 8, 6, 4

p = np.random.uniform(0.05, 0.35, size=n).tolist()
u = np.random.uniform(1, 10, size=n).tolist()

print(f'Instance: n={n}, B={B}, G={G}')
print(f'p = {["{:.3f}".format(pi) for pi in p]}')
print(f'u = {["{:.1f}".format(ui) for ui in u]}')
print()

K_values = list(range(0, B + 1))
eu_hybrid = []
time_hybrid = []

for K in K_values:
    t0 = time.time()
    tree_h, eu_h = hybrid_greedy_bruteforce(p, u, B, G, greedy_steps=K)
    elapsed = time.time() - t0
    eu_hybrid.append(eu_h)
    time_hybrid.append(elapsed)
    print(f'K={K}: EU={eu_h:.4f}, time={elapsed:.3f}s')

In [ ]:
# 3 plots: EU vs K, time vs K, gap vs K
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

ax1.plot(K_values, eu_hybrid, 'o-', color='#2980b9', linewidth=2, markersize=8)
ax1.axhline(y=eu_hybrid[0], color='#27ae60', linestyle='--',
            label=f'Full DP (K=0): {eu_hybrid[0]:.4f}')
ax1.set_xlabel('K (greedy steps)')
ax1.set_ylabel('Expected Utility')
ax1.set_title('EU vs K')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(K_values, time_hybrid, 'o-', color='#e74c3c', linewidth=2, markersize=8)
ax2.set_xlabel('K (greedy steps)')
ax2.set_ylabel('Time (seconds)')
ax2.set_title('Runtime vs K')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

gaps_k = [eu_hybrid[0] - eu for eu in eu_hybrid]
ax3.plot(K_values, gaps_k, 'o-', color='#8e44ad', linewidth=2, markersize=8)
ax3.set_xlabel('K (greedy steps)')
ax3.set_ylabel('Gap from Optimal (EU loss)')
ax3.set_title('Optimality Gap vs K')
ax3.grid(True, alpha=0.3)

plt.suptitle(f'Hybrid Solver: n={n}, B={B}, G={G}', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Branch value estimation after K=2 greedy steps
n, B, G = 8, 6, 4

# Run 2 greedy steps to get an intermediate state
tree_k2, eu_k2 = hybrid_greedy_bruteforce(p, u, B, G, greedy_steps=2)

# Get posteriors and cleared mask at the root after 2 steps
# Walk down the r=0 branch (best case) for illustration
node = tree_k2
path_desc = ['Root']
for depth in range(2):
    if node.get('terminal') or 'children' not in node:
        break
    # Take the r=0 branch if it exists, else first available
    children = node['children']
    r_choice = 0 if 0 in children else min(children.keys())
    node = children[r_choice]
    path_desc.append(f'r={r_choice}')

posteriors = node.get('posteriors', list(p))
cleared = node.get('cleared', 0)
remaining_B = B - 2  # 4 tests left

lb, ub = estimate_branch_value(posteriors, u, remaining_B, G, cleared, n)

print(f'After K=2 steps, following path: {", ".join(path_desc)}')
print(f'Cleared: {mask_str(cleared, n)}')
print(f'Posteriors: {["{:.3f}".format(pi) for pi in posteriors]}')
print(f'Remaining budget: {remaining_B}')
print(f'Branch value bounds: [{lb:.4f}, {ub:.4f}]')
print(f'Bound width: {ub - lb:.4f}')

In [ ]:
# Comparison table: Full DP vs Hybrid(K=2) vs Full Greedy
n, B, G = 8, 6, 4

t0 = time.time()
tree_full_dp, eu_full_dp = hybrid_greedy_bruteforce(p, u, B, G, greedy_steps=0)
time_full_dp = time.time() - t0

t0 = time.time()
tree_hybrid2, eu_hybrid2 = hybrid_greedy_bruteforce(p, u, B, G, greedy_steps=2)
time_hybrid2 = time.time() - t0

t0 = time.time()
tree_full_gr, eu_full_gr = hybrid_greedy_bruteforce(p, u, B, G, greedy_steps=B)
time_full_gr = time.time() - t0

print(f'{"Method":<20} {"EU":>10} {"Gap":>10} {"Time (s)":>10}')
print('-' * 52)
print(f'{"Full DP (K=0)":<20} {eu_full_dp:>10.4f} {0:>10.4f} {time_full_dp:>10.3f}')
print(f'{"Hybrid (K=2)":<20} {eu_hybrid2:>10.4f} '
      f'{eu_full_dp - eu_hybrid2:>10.4f} {time_hybrid2:>10.3f}')
print(f'{"Full Greedy (K=B)":<20} {eu_full_gr:>10.4f} '
      f'{eu_full_dp - eu_full_gr:>10.4f} {time_full_gr:>10.3f}')

In [ ]:
# Budget allocation analogy: n=12, B=10, K_greedy=8
# Larger instance where full DP is expensive
np.random.seed(123)
n_big, B_big, G_big = 12, 10, 4

p_big = np.random.uniform(0.05, 0.3, size=n_big).tolist()
u_big = np.random.uniform(1, 10, size=n_big).tolist()

print(f'Large instance: n={n_big}, B={B_big}, G={G_big}')
print(f'Full DP would require solving over {2**n_big} infection profiles -- very slow.')
print()

K_greedy_big = 8  # 8 greedy steps, then 2 DP steps on reduced subproblem

t0 = time.time()
tree_big_hybrid, eu_big_hybrid = hybrid_greedy_bruteforce(
    p_big, u_big, B_big, G_big, greedy_steps=K_greedy_big)
time_big_hybrid = time.time() - t0

t0 = time.time()
tree_big_greedy, eu_big_greedy = hybrid_greedy_bruteforce(
    p_big, u_big, B_big, G_big, greedy_steps=B_big)
time_big_greedy = time.time() - t0

print(f'Hybrid (K={K_greedy_big}): EU = {eu_big_hybrid:.4f}, '
      f'time = {time_big_hybrid:.3f}s')
print(f'Full Greedy:     EU = {eu_big_greedy:.4f}, '
      f'time = {time_big_greedy:.3f}s')
print(f'Improvement from DP tail: '
      f'{eu_big_hybrid - eu_big_greedy:.4f}')

---
## Block 5: Infection-Aware Greedy

The standard greedy maximizes P(r=0) * sum(u_i), which focuses on clearing.
The **infection-aware** score blends this with **expected information gain**
(entropy reduction):

$$\text{score}(t) = \alpha \cdot P(r=0) \sum u_i + (1-\alpha) \cdot \text{InfoGain}(t)$$

When alpha=1 we get pure clearing, when alpha=0 we get pure information
gathering.  Intermediate values balance exploration and exploitation.

In [ ]:
# --- Block 5: Infection-Aware Greedy ---
n, B, G = 6, 4, 3
p = [0.3] * n
u = [1.0] * n

alpha_values = [0.0, 0.25, 0.5, 0.75, 1.0]
eu_alpha = []
trees_alpha = []

for alpha in alpha_values:
    # Build a score function that captures alpha
    def make_score_fn(a):
        def score_fn(cp, u_arg, G_arg, n_arg, cleared_mask):
            return _infection_aware_best_pool(
                cp, u_arg, G_arg, n_arg, cleared_mask, alpha=a)
        return score_fn
    
    tree_a, eu_a = hybrid_greedy_bruteforce(
        p, u, B, G, greedy_steps=B,
        greedy_score_fn=make_score_fn(alpha))
    eu_alpha.append(eu_a)
    trees_alpha.append(tree_a)
    print(f'alpha={alpha:.2f}: EU = {eu_a:.4f}')

# Also get optimal for reference
val_opt_b5, _ = solve_optimal_dapts(p, u, B, G)
print(f'\nOptimal DP: EU = {val_opt_b5:.4f}')

In [ ]:
# Plot EU vs alpha
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(alpha_values, eu_alpha, 'o-', color='#8e44ad', linewidth=2, markersize=10)
ax.axhline(y=val_opt_b5, color='#27ae60', linestyle='--', linewidth=2,
           label=f'Optimal DP: {val_opt_b5:.4f}')
ax.fill_between(alpha_values, eu_alpha, alpha=0.15, color='#8e44ad')
ax.set_xlabel('alpha (0=info gain, 1=clearing)')
ax.set_ylabel('Expected Utility')
ax.set_title(f'Infection-Aware Greedy: EU vs alpha (n={n}, B={B}, G={G})')
ax.legend()
ax.grid(True, alpha=0.3)

for a, v in zip(alpha_values, eu_alpha):
    ax.annotate(f'{v:.3f}', (a, v), textcoords='offset points',
                xytext=(0, 12), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Show trees for best and worst alpha
best_idx = int(np.argmax(eu_alpha))
worst_idx = int(np.argmin(eu_alpha))

print(f'Best alpha:  {alpha_values[best_idx]:.2f} (EU={eu_alpha[best_idx]:.4f})')
print(f'Worst alpha: {alpha_values[worst_idx]:.2f} (EU={eu_alpha[worst_idx]:.4f})')

display(render_side_by_side(
    trees_alpha[best_idx], trees_alpha[worst_idx], n,
    title_a=f'Best: alpha={alpha_values[best_idx]} (EU={eu_alpha[best_idx]:.3f})',
    title_b=f'Worst: alpha={alpha_values[worst_idx]} (EU={eu_alpha[worst_idx]:.3f})',
    show_posteriors=False,
    max_depth=3,
))

In [ ]:
# Info gain visualization: bar chart of clearing vs info gain for candidate pools
n, B, G = 6, 4, 3
p = [0.3] * n
u = [1.0] * n
cleared_mask = 0

# Enumerate all candidate pools
active_mask, _ = compute_active_mask(p, cleared_mask, n)
pools = all_pools_from_mask(active_mask, G, include_empty=False)

# Limit to top pools for readability
pool_data = []
for pool in pools:
    pool_indices = indices_from_mask(pool, n)
    prob_clear = 1.0
    for i in pool_indices:
        prob_clear *= (1.0 - p[i])
    clearing_val = prob_clear * sum(u[i] for i in pool_indices
                                    if not (cleared_mask >> i & 1))
    ig = expected_info_gain(pool, p, n)
    pool_data.append({
        'pool': pool, 'pool_str': mask_str(pool, n),
        'clearing': clearing_val, 'info_gain': ig,
    })

# Sort by combined score and take top 15
pool_data.sort(key=lambda x: x['clearing'] + x['info_gain'], reverse=True)
pool_data = pool_data[:15]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(pool_data))
width = 0.35

clearing_vals = [d['clearing'] for d in pool_data]
ig_vals = [d['info_gain'] for d in pool_data]
pool_labels = [d['pool_str'] for d in pool_data]

ax.bar(x - width/2, clearing_vals, width, label='P(r=0) * sum(u_i)',
       color='#27ae60', alpha=0.8, edgecolor='black')
ax.bar(x + width/2, ig_vals, width, label='Expected Info Gain (bits)',
       color='#3498db', alpha=0.8, edgecolor='black')

ax.set_xlabel('Pool')
ax.set_ylabel('Score')
ax.set_title(f'Clearing Value vs Information Gain -- n={n}, p=0.3')
ax.set_xticks(x)
ax.set_xticklabels(pool_labels, rotation=45, ha='right', fontsize=8)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Hybrid connection: standard vs infection-aware Phase 1
n, B, G = 6, 4, 3
p = [0.3] * n
u = [1.0] * n
K_greedy = 2  # 2 greedy steps, then DP

# Standard greedy Phase 1
t0 = time.time()
tree_std, eu_std = hybrid_greedy_bruteforce(
    p, u, B, G, greedy_steps=K_greedy)
time_std = time.time() - t0

# Infection-aware Phase 1 (alpha=0.5)
def ia_score_fn(cp, u_arg, G_arg, n_arg, cm):
    return _infection_aware_best_pool(cp, u_arg, G_arg, n_arg, cm, alpha=0.5)

t0 = time.time()
tree_ia, eu_ia = hybrid_greedy_bruteforce(
    p, u, B, G, greedy_steps=K_greedy,
    greedy_score_fn=ia_score_fn)
time_ia = time.time() - t0

print(f'Hybrid (K={K_greedy}) with standard greedy: EU = {eu_std:.4f} '
      f'(time: {time_std:.3f}s)')
print(f'Hybrid (K={K_greedy}) with infection-aware: EU = {eu_ia:.4f} '
      f'(time: {time_ia:.3f}s)')
print(f'Difference: {eu_ia - eu_std:+.4f}')

display(render_side_by_side(
    tree_std, tree_ia, n,
    title_a=f'Standard Phase 1 (EU={eu_std:.3f})',
    title_b=f'Infection-Aware Phase 1 (EU={eu_ia:.3f})',
    show_posteriors=False,
    max_depth=3,
))

In [ ]:
# "Hunting infecteds" example
# One near-certain infected (agent 0), rest uncertain but more valuable
n_hunt = 5
B_hunt, G_hunt = 3, 3
p_hunt = [0.95, 0.3, 0.3, 0.3, 0.3]
u_hunt = [1.0, 2.0, 2.0, 2.0, 2.0]

print(f'"Hunting Infecteds" scenario: n={n_hunt}, B={B_hunt}, G={G_hunt}')
print(f'Agent 0: near-certain infected (p=0.95), low value (u=1)')
print(f'Agents 1-4: uncertain (p=0.3), higher value (u=2)')
print()

# Optimal
val_hunt_opt, pol_hunt_opt = solve_optimal_dapts(
    p_hunt, u_hunt, B_hunt, G_hunt)
tree_hunt_opt = extract_tree(pol_hunt_opt, p_hunt, u_hunt, n_hunt)

# Standard greedy
tree_hunt_std, eu_hunt_std = hybrid_greedy_bruteforce(
    p_hunt, u_hunt, B_hunt, G_hunt, greedy_steps=B_hunt)

# Infection-aware (alpha=0.3 to favor info gain)
def ia_hunt_fn(cp, u_arg, G_arg, n_arg, cm):
    return _infection_aware_best_pool(cp, u_arg, G_arg, n_arg, cm, alpha=0.3)

tree_hunt_ia, eu_hunt_ia = hybrid_greedy_bruteforce(
    p_hunt, u_hunt, B_hunt, G_hunt, greedy_steps=B_hunt,
    greedy_score_fn=ia_hunt_fn)

print(f'Optimal:          EU = {val_hunt_opt:.4f}')
print(f'Standard greedy:  EU = {eu_hunt_std:.4f}')
print(f'Infection-aware:  EU = {eu_hunt_ia:.4f}')
print()

group_colors_hunt = {0: 'red'}  # highlight the near-certain infected
group_colors_hunt.update({i: '#2980b9' for i in range(1, 5)})

display(render_tree_series(
    [tree_hunt_opt, tree_hunt_std, tree_hunt_ia], n_hunt,
    titles=[f'Optimal (EU={val_hunt_opt:.3f})',
            f'Standard Greedy (EU={eu_hunt_std:.3f})',
            f'Infection-Aware a=0.3 (EU={eu_hunt_ia:.3f})'],
    max_per_row=3,
    group_colors=group_colors_hunt,
    show_posteriors=False,
    max_depth=3,
))

print('\nKey insight: the infection-aware greedy may include agent 0 in pools')
print('strategically -- not to clear it (unlikely), but to gain information')
print('about the other agents via the count.')

---

## Summary

This notebook explored pooled testing decision trees from multiple angles:

1. **Block 0**: Simple cases showed how value, prevalence, budget, and pool size
   each shape the optimal tree structure.

2. **Block 1**: Heterogeneous populations (VIP vs common) reveal that the optimizer
   prioritizes high-value individuals while greedy may not.

3. **Block 2**: Binary search is a natural heuristic but the optimal strategy
   can exploit augmented information more flexibly.

4. **Block 3**: The greedy-to-optimal gap varies across instances; counting-based
   greedy can outperform sequential greedy by leveraging cross-test information.

5. **Block 4**: The hybrid solver provides a smooth tradeoff between computation
   time and solution quality.

6. **Block 5**: Infection-aware scoring balances clearing and information gathering,
   which is especially valuable when some agents are near-certain infected.